In [ ]:
import numpy as np
from multiprocess import Process, Queue, Event
import cv2
import yt_dlp

In [ ]:
#one time model export and conversion
from ultralytics import YOLO
model = YOLO("yolo26n.pt")
model.export(format="openvino" int8=True)

In [ ]:
def get_stream_url(youtube_url):
    # Using 'best[ext=mp4]' ensures better compatibility with OpenCV's FFmpeg backend
    ydl_opts = {"format": "best[ext=mp4]", "quiet": True}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=False)
            return info['url']
    except Exception as e:
        print(f"Error extracting {youtube_url}: {e}")
        return None

In [ ]:
def stream_worker(source_id, model_path, output_queue, stop_event):
    """
    Independent process for a single camera stream.
    """
    import cv2
    from ultralytics import YOLO
    import time

    # CRITICAL: Instantiate model INSIDE the process
    model = YOLO(model_path)
    cap = cv2.VideoCapture(source_id)

    fps=0
    frame_count=0
    t1=time.time()

    while not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            break

        start=time.time()

        # Inference with unique tracker for this stream
        #original model inference
        #results = model.track(frame, persist=True, verbose=False, device="intel:gpu")[0]
        results = model.track(
            frame,
            persist=True,
            verbose=False,
            device="intel:gpu",
            conf=0.45,      # Ignore low-confidence noise (HUGE speed gain)
            iou=0.5,        # Stricter overlap suppression
            max_det=50,     # Cap detections to 50 objects per stream
            vid_stride=2    # (Optional) Set to 2 to skip every other frame for massive FPS boost
        )[0]

        annotated_frame = results.plot()

        diff=time.time()-start
        if diff>0:
            current_fps=1.0/diff
            fps=(fps*0.9)+(current_fps*0.1) if fps>0 else current_fps

        # Label the frame with its source ID
        label=f"Stream:{source_id[:15]}... | FPS: {fps:.1f}"
        #cv2.putText(annotated_frame, f"Cam: {source_id}", (20, 40),
        #            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(annotated_frame, label, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        # Manage queue size to prevent memory buildup
        if output_queue.full():
            try: output_queue.get_nowait()
            except: pass
        output_queue.put((source_id, annotated_frame))

    cap.release()

if __name__ == "__main__":
    # Define your sources (0 for webcam, or RTSP strings)
    youtube_links = [
        #"https://www.youtube.com/watch?v=8H3nRCFVR6Y",
        "https://www.youtube.com/watch?v=glJu8snzi78",
        "https://www.youtube.com/watch?v=gFRtAAmiFbE",
        "https://www.youtube.com/watch?v=DjdUEyjx8GM",
        "https://www.youtube.com/watch?v=V1K18SNTUM8"
    ]

    print("extracting vid urls")
    sources = [get_stream_url(link) for link in youtube_links]
    sources = [s for s in sources if s is not None]
    #sources = [stream_url]  # Example: Two local webcams
    if not sources:
        print("error")
        exit()

    #ensure you are pointing to the right openvino converted model path
    model_path = "yolo26n_int8_openvino_model/"

    output_queue = Queue(maxsize=len(sources) * 4)
    stop_event = Event()
    processes = []

    # Start a process for each camera
    for src in sources:
        p = Process(target=stream_worker, args=(src, model_path, output_queue, stop_event))
        p.start()
        processes.append(p)

    # Main Thread: Single GUI Window
    frames_dict = {}
    target_size = (640, 360)

    try:
        while True:
            # Collect latest frames from the queue
            while not output_queue.empty():
                src_id, frame = output_queue.get()
                #frames_dict[src_id] = frame
                frames_dict[src_id] = cv2.resize(frame, target_size)

            if frames_dict:
                #original vid stitch
                # Advanced: Stitch frames into a grid for one window
                #all_frames = list(frames_dict.values())
                # Simple horizontal stack (adjust for grid as needed)
                #grid_display = np.hstack(all_frames) if len(all_frames) > 1 else all_frames[0]

                #stitch 4x4 grid
                sorted_labels = sorted(frames_dict.keys())
                all_frames = [frames_dict[label] for label in sorted_labels]

                num_streams = len(all_frames)
                if num_streams == 1:
                    grid_display = all_frames[0]
                elif num_streams == 2:
                    grid_display = np.hstack(all_frames)
                else:
                    # Create 2x2 grid
                    while len(all_frames) < 4:
                        all_frames.append(np.zeros((target_size[1], target_size[0], 3), dtype=np.uint8))

                    row1 = np.hstack(all_frames[0:2])
                    row2 = np.hstack(all_frames[2:4])
                    grid_display = np.vstack([row1, row2])

                cv2.imshow("Multi-Stream YOLO Tracking", grid_display)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        stop_event.set()
        for p in processes:
            p.terminate()
        cv2.destroyAllWindows()


extracting vid urls
